### 01. Environment Verification & Hardware Setup

This initial cell validates that we are running on the correct Python version inside our virtual environment (`venv`), checks PyTorch availability, and automatically determines whether processing will run on a GPU or using local CPU optimizations.

In [6]:
# ==========================================
# 01. Environment Verification & Setup
# ==========================================

import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")

# Device detection
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Active Execution Device: {device}")

if device.type == "cpu":
    print("-> Note: Running on local CPU.")
else:
    print("-> GPU detected and ready for acceleration.")

Python Version: 3.11.0
PyTorch Version: 2.13.0+cpu
Active Execution Device: cpu
-> Note: Running on local CPU.


### Cell 02 (Updated): Downloading and Loading the E-Commerce Dataset from Hugging Face
This script downloads the Women's Clothing E-Commerce Reviews dataset directly from the Hugging Face repository, saves it to the local `data/raw/` folder, and performs an initial structural inspection (dimensions, columns, and missing values).

In [7]:
# ==========================================
# 02. Download and Load Raw Dataset
# ==========================================

import os
import pandas as pd
import requests

# Define project-relative path and file target
raw_dir = "../data/raw"

os.makedirs(raw_dir, exist_ok=True)

file_path = os.path.join(
    raw_dir,
    "womens_clothing_reviews_raw.csv"
)

dataset_url = (
    "https://huggingface.co/datasets/Censius-AI/"
    "ECommerce-Women-Clothing-Reviews/resolve/main/"
    "Womens%20Clothing%20E-Commerce%20Reviews.csv"
)

# Download the dataset if it doesn't exist locally
if not os.path.exists(file_path):
    print("Downloading dataset from Hugging Face...")

    response = requests.get(dataset_url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    print(
        f"Dataset successfully downloaded and saved to: "
        f"{file_path}"
    )

else:
    print(
        f"Dataset already exists locally at: "
        f"{file_path}"
    )

# Load the dataset
df = pd.read_csv(file_path)

# Display dataset dimensions and initial rows
print("\n--- Dataset Info ---")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

print("\n--- Column Names ---")
print(df.columns.tolist())

print("\n--- First 3 Rows ---")
display(df.head(3))

print("\n--- Missing Values Check ---")
print(df.isnull().sum())

Dataset already exists locally at: ../data/raw\womens_clothing_reviews_raw.csv

--- Dataset Info ---
Total Rows: 23486
Total Columns: 11

--- Column Names ---
['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']

--- First 3 Rows ---


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses



--- Missing Values Check ---
Unnamed: 0                    0
Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64


### Cell 03 (Optimized Three-Class Sentiment Pipeline): Advanced Text Cleaning & Balancing
This script loads the raw dataset, removes index artifacts, applies advanced text cleaning (handling HTML, URLs, length truncation, and unnecessary punctuation while preserving sentiment markers like ! and ?), maps ratings to a 3-class target, and creates a balanced dataset.

In [8]:
# ==========================================
# 03. Text Cleaning and Initial Sentiment Labels
# ==========================================

import os
import re
import pandas as pd

# Define project-relative paths
raw_file_path = "../data/raw/womens_clothing_reviews_raw.csv"
cleaned_file_path = "../data/processed/womens_clothing_reviews_cleaned.csv"

# Ensure processed directory exists
os.makedirs(
    os.path.dirname(cleaned_file_path),
    exist_ok=True
)

# Load raw dataset
print(f"Loading raw dataset from: {raw_file_path}")

df = pd.read_csv(raw_file_path)

# Drop index artifacts if present
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print(f"Initial raw rows: {df.shape[0]}")


# ------------------------------------------
# 1. Text Cleaning Function
# ------------------------------------------

def clean_text_advanced(text, max_chars=400):

    if not isinstance(text, str):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags and URLs
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(
        r"https?://\S+|www\.\S+",
        "",
        text
    )

    # Keep alphanumeric characters, spaces,
    # and core sentiment punctuation
    text = re.sub(
        r"[^a-z0-9\s!?]",
        " ",
        text
    )

    # Truncate long reviews
    if len(text) > max_chars:
        text = text[:max_chars]

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ------------------------------------------
# 2. Clean Reviews
# ------------------------------------------

print("Applying text cleaning and length truncation...")

df_cleaned = df.dropna(
    subset=["Review Text", "Rating"]
).copy()

df_cleaned["Review Text"] = df_cleaned[
    "Review Text"
].apply(
    lambda x: clean_text_advanced(
        x,
        max_chars=400
    )
)

# Remove empty or extremely short reviews
df_cleaned = df_cleaned[
    df_cleaned["Review Text"].str.len() > 15
].reset_index(drop=True)

print(
    f"Rows after cleaning: {len(df_cleaned)}"
)


# ------------------------------------------
# 3. Initial Sentiment Mapping
# ------------------------------------------

# 1-2 stars -> Negative (0)
# 3 stars   -> Neutral (1)
# 4-5 stars -> Positive (2)

def map_initial_sentiment(rating):

    if rating <= 2:
        return 0

    elif rating == 3:
        return 1

    else:
        return 2


df_cleaned["sentiment_target"] = df_cleaned[
    "Rating"
].apply(
    map_initial_sentiment
)


# ------------------------------------------
# 4. Display Initial Distribution
# ------------------------------------------

print("\n--- Initial 3-Class Distribution ---")

print(
    df_cleaned["sentiment_target"]
    .value_counts()
    .sort_index()
)

print("\n--- Sentiment Labels ---")
print("0 = Negative")
print("1 = Neutral")
print("2 = Positive")


# ------------------------------------------
# 5. Save Complete Cleaned Dataset
# ------------------------------------------

df_cleaned.to_csv(
    cleaned_file_path,
    index=False
)

print(
    f"\nCleaned dataset successfully created "
    f"and saved to: {cleaned_file_path}"
)

Loading raw dataset from: ../data/raw/womens_clothing_reviews_raw.csv
Initial raw rows: 23486
Applying text cleaning and length truncation...
Rows after cleaning: 22632

--- Initial 3-Class Distribution ---
sentiment_target
0     2370
1     2823
2    17439
Name: count, dtype: int64

--- Sentiment Labels ---
0 = Negative
1 = Neutral
2 = Positive

Cleaned dataset successfully created and saved to: ../data/processed/womens_clothing_reviews_cleaned.csv


### Cell 04: Filter Truly Neutral Reviews (The Smart Filtering Approach)
This script loads the raw dataset, isolates 3-star candidates, and runs inference using cardiffnlp/twitter-roberta-base-sentiment-latest via Softmax probabilities to isolate genuinely neutral texts.

In [9]:
# ==========================================
# 04. Neutral Curation with Auxiliary RoBERTa
# ==========================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Select only 3-star reviews
df_neutral_candidates = df_cleaned[
    df_cleaned["Rating"] == 3
].copy()

print(
    f"3-star reviews available for neutral curation: "
    f"{len(df_neutral_candidates)}"
)


# ------------------------------------------
# Load Auxiliary RoBERTa Model
# ------------------------------------------

auxiliary_model_name = (
    "cardiffnlp/twitter-roberta-base-sentiment-latest"
)

tokenizer_aux = AutoTokenizer.from_pretrained(
    auxiliary_model_name
)

model_aux = AutoModelForSequenceClassification.from_pretrained(
    auxiliary_model_name
)

model_aux.to(device)
model_aux.eval()


# ------------------------------------------
# Classify 3-Star Reviews
# ------------------------------------------

texts = df_neutral_candidates[
    "Review Text"
].tolist()

predicted_labels = []

batch_size = 32

print(
    "Classifying 3-star reviews with "
    "auxiliary RoBERTa..."
)

for i in range(
    0,
    len(texts),
    batch_size
):

    batch_texts = texts[
        i:i + batch_size
    ]

    inputs = tokenizer_aux(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model_aux(
            **inputs
        )

    predictions = torch.argmax(
        outputs.logits,
        dim=1
    ).cpu().tolist()

    predicted_labels.extend(
        predictions
    )


# Auxiliary model labels:
# 0 = Negative
# 1 = Neutral
# 2 = Positive

df_neutral_candidates[
    "auxiliary_sentiment"
] = predicted_labels


# ------------------------------------------
# Keep Only Auxiliary-Neutral Reviews
# ------------------------------------------

df_neutral_clean = df_neutral_candidates[
    df_neutral_candidates[
        "auxiliary_sentiment"
    ] == 1
].copy()


# Final training label for these reviews
df_neutral_clean[
    "sentiment_target"
] = 1


# Remove auxiliary prediction column
df_neutral_clean = df_neutral_clean.drop(
    columns=["auxiliary_sentiment"]
)


df_neutral_clean = df_neutral_clean.reset_index(
    drop=True
)


# ------------------------------------------
# Results
# ------------------------------------------

print(
    f"\n3-star reviews classified as Neutral: "
    f"{len(df_neutral_clean)}"
)

print("\n--- Curated Neutral Distribution ---")

print(
    df_neutral_clean["sentiment_target"]
    .value_counts()
)

print("\nNeutral curation completed successfully.")

3-star reviews available for neutral curation: 2823


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21122.85it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Classifying 3-star reviews with auxiliary RoBERTa...

3-star reviews classified as Neutral: 390

--- Curated Neutral Distribution ---
sentiment_target
1    390
Name: count, dtype: int64

Neutral curation completed successfully.


### Cell 05: Dataset Balancing & Final Persistence (3 Classes)
We ensure all three sentiment classes (0.0: Negative, 1.0: Neutral, 2.0: Positive) are correctly isolated, balanced to equal sizes (293 samples each), and persisted to the processed directory.

In [10]:
# ==========================================
# 05. Build Final Curated Sentiment Dataset
# ==========================================

import os
import pandas as pd

# Define project-relative output path
output_path = (
    "../data/processed/"
    "womens_clothing_roberta_reviews.csv"
)

# Ensure processed directory exists
os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)


# ------------------------------------------
# 1. Select Negative and Positive Reviews
# ------------------------------------------

# Negative:
# Initial labels from ratings 1-2
df_negative = df_cleaned[
    df_cleaned["sentiment_target"] == 0
].copy()

# Positive:
# Initial labels from ratings 4-5
df_positive = df_cleaned[
    df_cleaned["sentiment_target"] == 2
].copy()


# ------------------------------------------
# 2. Combine All Final Classes
# ------------------------------------------

# Neutral:
# Only 3-star reviews classified as Neutral
# by the auxiliary CardiffNLP RoBERTa model.

df_final = pd.concat(
    [
        df_negative,
        df_neutral_clean,
        df_positive
    ],
    ignore_index=True
)


# ------------------------------------------
# 3. Shuffle Final Dataset
# ------------------------------------------

df_final = df_final.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# ------------------------------------------
# 4. Display Final Distribution
# ------------------------------------------

print(
    "\n--- Final Dataset Distribution "
    "Before Balancing ---"
)

print(
    df_final["sentiment_target"]
    .value_counts()
    .sort_index()
)

print(
    f"\nTotal reviews before balancing: "
    f"{len(df_final)}"
)


# ------------------------------------------
# 5. Persist Final Curated Dataset
# ------------------------------------------

df_final.to_csv(
    output_path,
    index=False
)

print(
    "\nFinal RoBERTa-curated dataset successfully "
    f"saved to: {output_path}"
)


--- Final Dataset Distribution Before Balancing ---
sentiment_target
0     2370
1      390
2    17439
Name: count, dtype: int64

Total reviews before balancing: 20199

Final RoBERTa-curated dataset successfully saved to: ../data/processed/womens_clothing_roberta_reviews.csv


### Cell 06 (RoBERTa Version): Tokenization & Training Preparation
We load our RoBERTa-curated balanced dataset, split it into stratified training and validation sets, and tokenize the texts using RobertaTokenizerFast.

In [11]:
# ==========================================
# 06. Balance Final Sentiment Dataset
# ==========================================

import os
import pandas as pd

# Define project-relative output path
output_path = (
    "../data/processed/"
    "womens_clothing_balanced_roberta_reviews.csv"
)

# Ensure processed directory exists
os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)


# ------------------------------------------
# 1. Check Current Class Distribution
# ------------------------------------------

print("\n--- Class Distribution Before Balancing ---")

print(
    df_final["sentiment_target"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------
# 2. Separate Sentiment Classes
# ------------------------------------------

df_negative = df_final[
    df_final["sentiment_target"] == 0
].copy()

df_neutral = df_final[
    df_final["sentiment_target"] == 1
].copy()

df_positive = df_final[
    df_final["sentiment_target"] == 2
].copy()


# ------------------------------------------
# 3. Determine Maximum Balanced Size
# ------------------------------------------

min_count = min(
    len(df_negative),
    len(df_neutral),
    len(df_positive)
)

print(
    f"\nBalanced samples per class: {min_count}"
)


# ------------------------------------------
# 4. Sample Each Class Equally
# ------------------------------------------

df_negative_balanced = df_negative.sample(
    n=min_count,
    random_state=42
)

df_neutral_balanced = df_neutral.sample(
    n=min_count,
    random_state=42
)

df_positive_balanced = df_positive.sample(
    n=min_count,
    random_state=42
)


# ------------------------------------------
# 5. Combine and Shuffle
# ------------------------------------------

df_balanced = pd.concat(
    [
        df_negative_balanced,
        df_neutral_balanced,
        df_positive_balanced
    ],
    ignore_index=True
)

df_balanced = df_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# ------------------------------------------
# 6. Verify Final Distribution
# ------------------------------------------

print(
    "\n--- Final Balanced Distribution ---"
)

print(
    df_balanced["sentiment_target"]
    .value_counts()
    .sort_index()
)

print(
    f"\nTotal balanced reviews: "
    f"{len(df_balanced)}"
)


# ------------------------------------------
# 7. Persist Final Training Dataset
# ------------------------------------------

df_balanced.to_csv(
    output_path,
    index=False
)

print(
    "\nFinal balanced RoBERTa training dataset "
    f"successfully saved to: {output_path}"
)


--- Class Distribution Before Balancing ---
sentiment_target
0     2370
1      390
2    17439
Name: count, dtype: int64

Balanced samples per class: 390

--- Final Balanced Distribution ---
sentiment_target
0    390
1    390
2    390
Name: count, dtype: int64

Total balanced reviews: 1170

Final balanced RoBERTa training dataset successfully saved to: ../data/processed/womens_clothing_balanced_roberta_reviews.csv


### Cell 07: Train / Validation Split + RoBERTa Tokenizer

In [12]:
# ==========================================
# 07. Train / Validation Split & Tokenization
# ==========================================

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer


# ------------------------------------------
# 1. Prepare Texts and Labels
# ------------------------------------------

texts = df_balanced["Review Text"].astype(str).tolist()

labels = df_balanced["sentiment_target"].astype(int).tolist()


# ------------------------------------------
# 2. Stratified Train / Validation Split
# ------------------------------------------

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print(
    f"Training samples: {len(train_texts)}"
)

print(
    f"Validation samples: {len(val_texts)}"
)


# ------------------------------------------
# 3. Initialize RoBERTa Tokenizer
# ------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    "roberta-base"
)


# ------------------------------------------
# 4. Tokenize Training and Validation Data
# ------------------------------------------

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding="max_length",
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding="max_length",
    max_length=128
)


print(
    "\nRoBERTa tokenization completed successfully."
)

Training samples: 936
Validation samples: 234

RoBERTa tokenization completed successfully.


In [ ]:
# Cell 7.1: Export Best Model for Inference & Dashboard
import os

# Define the definitive destination path for the clean, production-ready model
best_model_path = '../models/fine_tuned_roberta/best_model'
os.makedirs(best_model_path, exist_ok=True)

# Save the trained model and tokenizer directly, bypassing intermediate checkpoints
trainer.model.save_pretrained(best_model_path)
tokenizer.save_pretrained(best_model_path)

print(f"Best fine-tuned RoBERTa model successfully exported and saved at: {best_model_path}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.21it/s]

Best fine-tuned RoBERTa model successfully exported and saved at: ../models/fine_tuned_roberta/best_model


## Cell 08: Dataset, Metrics, Model & Trainer Setup

In [14]:
# ==========================================
# 08. Dataset, Metrics, Model & Trainer Setup
# ==========================================

import torch
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)


# ------------------------------------------
# 1. PyTorch Dataset
# ------------------------------------------

class ReviewDatasetRB(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(
                value[idx],
                dtype=torch.long
            )
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            int(self.labels[idx]),
            dtype=torch.long
        )

        return item


# Create training and validation datasets
train_dataset = ReviewDatasetRB(
    train_encodings,
    train_labels
)

val_dataset = ReviewDatasetRB(
    val_encodings,
    val_labels
)

print(
    f"Training dataset size: {len(train_dataset)}"
)

print(
    f"Validation dataset size: {len(val_dataset)}"
)


# ------------------------------------------
# 2. Evaluation Metrics
# ------------------------------------------

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ------------------------------------------
# 3. Load Base RoBERTa Model
# ------------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

print(
    "\nRoBERTa model initialized successfully."
)

print(
    "Classes: 3 "
    "(Negative=0, Neutral=1, Positive=2)"
)


# ------------------------------------------
# 4. Training Configuration
# ------------------------------------------

training_args = TrainingArguments(

    output_dir="../models/fine_tuned_roberta",

    num_train_epochs=5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=32,

    learning_rate=5e-5,

    warmup_steps=10,

    weight_decay=0.01,

    logging_steps=5,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True
)


# ------------------------------------------
# 5. Initialize Trainer
# ------------------------------------------

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)


print(
    "\nTrainer initialized successfully."
)

print(
    "Ready to start RoBERTa fine-tuning."
)

Training dataset size: 936
Validation dataset size: 234


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3281.16it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



RoBERTa model initialized successfully.
Classes: 3 (Negative=0, Neutral=1, Positive=2)

Trainer initialized successfully.
Ready to start RoBERTa fine-tuning.


## Cell 09: Fine-tuning

In [15]:
# ==========================================
# 09. RoBERTa Fine-Tuning
# ==========================================

print(
    "Starting RoBERTa fine-tuning on "
    "balanced review data..."
)

trainer.train()

print(
    "\nFine-tuning completed successfully!"
)

Starting RoBERTa fine-tuning on balanced review data...


c:\Users\con2m\Desktop\PORTFOLIO\sentiment_business_intelligence\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.667195,0.788337,0.670940,0.701419,0.670940,0.654894
2,0.523876,0.811755,0.692308,0.693143,0.692308,0.679280
3,0.430989,0.975767,0.713675,0.757131,0.713675,0.713700
4,0.166034,0.874941,0.747863,0.749408,0.747863,0.747921
5,0.155440,0.947144,0.739316,0.742431,0.739316,0.740617


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.04it/s]
c:\Users\con2m\Desktop\PORTFOLIO\sentiment_business_intelligence\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]
c:\Users\con2m\Desktop\PORTFOLIO\sentiment_business_intelligence\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.27it/s]
c:\Users\con2m\Desktop\PORTFOLIO\sentiment_business_intelligence\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().


Fine-tuning completed successfully!


## Cell 10: Export Best Model for Inference

In [16]:
# ==========================================
# 10. Export Best Model for Inference
# ==========================================

import os

# Define the definitive destination path
best_model_path = (
    "../models/fine_tuned_roberta/best_model"
)

# Ensure the destination directory exists
os.makedirs(
    best_model_path,
    exist_ok=True
)


# ------------------------------------------
# Save Best Fine-Tuned Model
# ------------------------------------------

trainer.model.save_pretrained(
    best_model_path
)

# Save the corresponding tokenizer
tokenizer.save_pretrained(
    best_model_path
)


print(
    "\nBest fine-tuned RoBERTa model successfully "
    f"exported to: {best_model_path}"
)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.45it/s]


Best fine-tuned RoBERTa model successfully exported to: ../models/fine_tuned_roberta/best_model


In [38]:
!pip install matplotlib

  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ---------------------------------------- 9.3/9.3 MB 58.0 MB/s  0:00:00
Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl (225 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 67.6 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)

   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------

## Cell 11: Detailed Model Evaluation

In [17]:
# ==========================================
# 11. Detailed Model Evaluation
# ==========================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

import numpy as np


# ------------------------------------------
# 1. Evaluate Validation Dataset
# ------------------------------------------

evaluation_results = trainer.evaluate()

print("\n--- Overall Evaluation Results ---")

for metric, value in evaluation_results.items():

    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")

    else:
        print(f"{metric}: {value}")


# ------------------------------------------
# 2. Generate Predictions
# ------------------------------------------

predictions_output = trainer.predict(
    val_dataset
)

predicted_labels = np.argmax(
    predictions_output.predictions,
    axis=-1
)

true_labels = np.array(
    val_labels
)


# ------------------------------------------
# 3. Classification Report
# ------------------------------------------

print("\n--- Classification Report ---")

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ],
        digits=4
    )
)


# ------------------------------------------
# 4. Confusion Matrix
# ------------------------------------------

conf_matrix = confusion_matrix(
    true_labels,
    predicted_labels
)

print("\n--- Confusion Matrix ---")

print(conf_matrix)

c:\Users\con2m\Desktop\PORTFOLIO\sentiment_business_intelligence\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.155440,0.874941,5,0.747863,0.749408,0.747863,0.747921



--- Overall Evaluation Results ---
eval_loss: 0.8749
eval_accuracy: 0.7479
eval_precision: 0.7494
eval_recall: 0.7479
eval_f1: 0.7479


c:\Users\con2m\Desktop\PORTFOLIO\sentiment_business_intelligence\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- Classification Report ---
              precision    recall  f1-score   support

    Negative     0.7059    0.7692    0.7362        78
     Neutral     0.6757    0.6410    0.6579        78
    Positive     0.8667    0.8333    0.8497        78

    accuracy                         0.7479       234
   macro avg     0.7494    0.7479    0.7479       234
weighted avg     0.7494    0.7479    0.7479       234


--- Confusion Matrix ---
[[60 14  4]
 [22 50  6]
 [ 3 10 65]]


---------------------

# Notebook 1 — Sentiment Model Training & Evaluation


## Overview

This notebook develops and evaluates a three-class sentiment classification model for customer reviews.

The objective is to transform raw customer reviews into reliable sentiment labels that will later serve as the sentiment analysis layer of the **Customer Reviews Intelligence** system.

### Complete Pipeline

**Raw Reviews → Text Cleaning → Initial Labels → Neutral Curation → Final Curated Dataset → Class Balancing → Train/Validation Split → RoBERTa Fine-Tuning → Evaluation → Model Export**

---

## 1. Dataset Preparation

The original Women's Clothing Reviews dataset was downloaded from Hugging Face and loaded into the notebook.

The review text was cleaned by:

- Removing missing review texts and ratings.
- Removing HTML tags and URLs.
- Converting text to lowercase.
- Removing unnecessary characters.
- Normalizing whitespace.
- Limiting review length to 400 characters.
- Removing empty or extremely short reviews.

The complete cleaned dataset was saved as:

`../data/processed/womens_clothing_reviews_cleaned.csv`

This dataset contains the full cleaned review population and will be used later for customer-review pattern discovery.

---

## 2. Initial Sentiment Definition

Initial sentiment labels were assigned according to the original star rating:

| Rating | Initial Sentiment |
|---|---|
| 1–2 stars | Negative |
| 3 stars | Neutral |
| 4–5 stars | Positive |

Three-star reviews were not automatically treated as reliable Neutral examples because they can contain mixed or ambiguous sentiment.

---

## 3. Neutral Sentiment Curation

Three-star reviews were evaluated using the auxiliary model:

**cardiffnlp/twitter-roberta-base-sentiment-latest**

Only reviews classified as **Neutral** by this auxiliary model were retained as Neutral examples for supervised training.

The auxiliary CardiffNLP model was used exclusively for **Neutral data curation**.

It is not the final sentiment classifier used by the application.

---

## 4. Final Curated Sentiment Dataset

The final sentiment dataset was constructed from:

- Negative reviews from ratings 1–2.
- Curated Neutral reviews from rating 3.
- Positive reviews from ratings 4–5.

The resulting dataset was shuffled and saved as:

`../data/processed/womens_clothing_roberta_reviews.csv`

This represents the final curated sentiment dataset before balancing.

---

## 5. Class Balancing

The final curated sentiment dataset was balanced by randomly sampling the same number of reviews from each sentiment class.

The three classes therefore have equal representation:

- Negative
- Neutral
- Positive

The final balanced dataset was saved as:

`../data/processed/womens_clothing_balanced_roberta_reviews.csv`

This balanced dataset was used exclusively for supervised sentiment model training.

The full cleaned dataset remains separate and is not artificially balanced because it will later be used for customer-review pattern discovery.

---

## 6. Train / Validation Split

The balanced sentiment dataset was divided using a stratified split:

- **80% Training**
- **20% Validation**

Stratification preserved the class distribution across both subsets.

The reviews were tokenized using the `roberta-base` tokenizer with a maximum sequence length of 128 tokens.

The tokenized training and validation data were kept in notebook memory and were not saved as separate files.

---

## 7. RoBERTa Fine-Tuning

A `roberta-base` transformer model was fine-tuned for three-class sentiment classification:

- **0 = Negative**
- **1 = Neutral**
- **2 = Positive**

### Training Configuration

- Maximum epochs: **5**
- Training batch size: **16**
- Validation batch size: **32**
- Learning rate: **5e-5**
- Warmup steps: **10**
- Weight decay: **0.01**
- Evaluation: every epoch
- Checkpoint saving: every epoch
- Early stopping patience: **2 epochs**
- Best model selection metric: **Weighted F1**

The best validation performance was achieved at **Epoch 4**.

---

## 8. Model Performance

The best model achieved:

| Metric | Result |
|---|---:|
| Accuracy | **74.79%** |
| Weighted F1 | **0.7479** |
| Macro F1 | **0.7479** |

### Performance by Sentiment Class

| Sentiment | Precision | Recall | F1 |
|---|---:|---:|---:|
| Negative | 70.6% | 76.9% | 73.6% |
| Neutral | 67.6% | 64.1% | 65.8% |
| Positive | 86.7% | 83.3% | 85.0% |

The model performs strongest on **Positive** reviews, while **Neutral** is the most challenging class.

This is expected because Neutral reviews frequently contain mixed, subtle, or ambiguous language.

Despite this challenge, the model provides meaningful classification performance across all three sentiment classes.

---

## 9. Confusion Matrix

The validation confusion matrix was:

| Actual / Predicted | Negative | Neutral | Positive |
|---|---:|---:|---:|
| **Negative** | 60 | 14 | 4 |
| **Neutral** | 22 | 50 | 6 |
| **Positive** | 3 | 10 | 65 |

The model correctly classified:

- **60 / 78** Negative reviews.
- **50 / 78** Neutral reviews.
- **65 / 78** Positive reviews.

The strongest classification performance was obtained for Positive reviews.

The main source of confusion was between **Negative and Neutral** sentiment, particularly for Neutral reviews.

---

## 10. Best Model Selection

Although training continued through Epoch 5, Epoch 4 produced the highest validation F1 score.

The Trainer was configured to restore the best model at the end of training using validation F1 as the selection metric.

Therefore, the exported model corresponds to the best-performing checkpoint rather than simply the final training epoch.

---

## 11. Model Export

The best fine-tuned RoBERTa model and its tokenizer were exported to:

`../models/fine_tuned_roberta/best_model`

This exported model is the definitive sentiment classifier intended for later inference in the **Customer Reviews Intelligence** application.

---

## 12. Notebook 1 Outcome

Notebook 1 successfully completed the supervised sentiment modeling stage of the project.

The final workflow produced:

- A cleaned review dataset.
- Curated sentiment labels.
- A balanced supervised training dataset.
- A stratified training/validation split.
- A fine-tuned three-class RoBERTa model.
- Detailed validation metrics.
- A confusion matrix.
- An exported best-performing model and tokenizer.

The final sentiment classifier achieved approximately **74.8% accuracy and 0.748 F1** on the balanced validation set.

This model now serves as the **sentiment analysis layer** of the broader Customer Reviews Intelligence system.

---

## 13. Next Stage — Notebook 2

Clustering and customer-review pattern discovery are intentionally handled separately from the supervised sentiment training performed in this notebook.

Notebook 2 will use the **full cleaned review dataset**, rather than the artificially balanced sentiment-training dataset.

### Next Pipeline

**Full Cleaned Reviews → Sentence-BERT Embeddings → UMAP → HDBSCAN → Customer Review Patterns → Business Insights**

The purpose of this stage is not to predict sentiment, but to discover naturally occurring themes and patterns within customer feedback.

This separation ensures that:

- Sentiment classification is trained on a balanced dataset.
- Pattern discovery reflects the natural distribution of all customer reviews.
- Sentiment analysis and pattern discovery remain technically and conceptually distinct.
```

## Cell 12: Semantic Embeddings & Unsupervised Clustering Strategy
We now implement semantic vectorization via Sentence-BERT (all-MiniLM-L6-v2), dimensionality reduction via UMAP, and density-based clustering via HDBSCAN to discover organic customer patterns and issues without manual hardcoding. Additionally, UMAP and HDBSCAN are configured to preserve projection and prediction capabilities, enabling a real-world production web application to classify newly arriving reviews via inference (umap.transform and hdbscan.approximate_predict) without retraining the entire pipeline.

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# 1. Load our cleaned and balanced dataset
df_balanced = pd.read_csv('../data/processed/balanced_reviews.csv')
texts = df_balanced['Review Text'].astype(str).tolist()

# 2. Generate semantic embeddings with Sentence-BERT
print("[*] Loading Sentence-BERT model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("[*] Generating dense vector embeddings...")
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=64)

# 3. Dimensionality reduction with UMAP (retaining transform capability for future production data)
print("[*] Reducing embedding dimensions with UMAP...")
umap_reducer = umap.UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
reduced_embeddings = umap_reducer.fit_transform(embeddings)

# 4. Density-based clustering with HDBSCAN
print("[*] Executing HDBSCAN clustering...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True  # Enables production-ready inference for new incoming reviews
)
cluster_labels = clusterer.fit_predict(reduced_embeddings)

# 5. Audit results
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)
print(f"[+] Clustering completed: {n_clusters} organic topic clusters discovered.")
print(f"[+] Outlier/Noise reviews (-1): {n_noise}")

c:\Users\con2m\Desktop\PORTFOLIO\sentiment_business_intelligence\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Loading Sentence-BERT model (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3226.97it/s]


[*] Generating dense vector embeddings...


Batches: 100%|██████████| 14/14 [00:12<00:00,  1.12it/s]
c:\Users\con2m\Desktop\PORTFOLIO\sentiment_business_intelligence\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[*] Reducing embedding dimensions with UMAP...
[*] Executing HDBSCAN clustering...
[+] Clustering completed: 10 organic topic clusters discovered.
[+] Outlier/Noise reviews (-1): 285


## Cell 13: Persistence of Spatial Models to the Vector Store
To maintain strict alignment with our project directory architecture, we serialize and persist the fitted UMAP reducer and HDBSCAN clusterer directly into our pre-established models/vector_store/ directory using joblib. This ensures that any future production interface (such as a Streamlit batch upload or automated API webhook) can instantly load these static models to project and classify incoming reviews via umap.transform and hdbscan.approximate_predict without disrupting historical data.

In [2]:
import joblib
import os

# 1. Target the pre-defined vector store directory in our project structure
vector_store_dir = '../models/vector_store'
os.makedirs(vector_store_dir, exist_ok=True)

# 2. Serialize and save the fitted UMAP reducer and HDBSCAN clusterer
joblib.dump(umap_reducer, os.path.join(vector_store_dir, 'umap_reducer.pkl'))
joblib.dump(clusterer, os.path.join(vector_store_dir, 'hdbscan_clusterer.pkl'))

print(f"[+] UMAP and HDBSCAN production artifacts successfully saved to: {vector_store_dir}")

[+] UMAP and HDBSCAN production artifacts successfully saved to: ../models/vector_store
